# Lightfielder for Resolve Studio
## Jupyter Link Example: Jupyter Resolve Studio.ipynb

Updated: 2026-09-06

An example that shows how to connect a [Jupyter Notebook](https://jupyter.org/) session to Resolve Studio v20-21+ using Python v3.6-3.15+.


In [24]:
# Load the BMD Resolve Studio Python scripting bindings
# Supports Python v3.6 to 3.15+

import sys, os, re, csv, datetime, math, json
import importlib.machinery, importlib.util
from pprint import pprint

import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

def FuScriptLib():
	lib_path = ""
	if sys.platform.startswith("darwin"):
		#lib_path = "/Applications/DaVinci Resolve 21/DaVinci Resolve.app/Contents/Libraries/Fusion/fusionscript.so"
		lib_path = "/Applications/DaVinci Resolve/DaVinci Resolve.app/Contents/Libraries/Fusion/fusionscript.so"
		#lib_path = "/Applications/Blackmagic Fusion 21/Fusion.app/Contents/Fusion/fusionscript.so"
		#lib_path = /Applications/Blackmagic Fusion 21 Render Node/Fusion Render Node.app/Contents/Fusion/fusionscript.so
	elif sys.platform.startswith("win"):
		#lib_path = "C:\\Program Files\\Blackmagic Design\\DaVinci Resolve 21\\fusionscript.dll"
		lib_path = "C:\\Program Files\\Blackmagic Design\\DaVinci Resolve\\fusionscript.dll"
		#lib_path = "C:\\Program Files\\Blackmagic Design\\Fusion 21\\fusionscript.dll"
		#lib_path = "C:\\Program Files\\Blackmagic Design\\Fusion Render Node 21\\fusionscript.dll"
	elif sys.platform.startswith("linux"):
		lib_path = "/opt/resolve/libs/Fusion/fusionscript.so"
		#lib_path = "/opt/BlackmagicDesign/Fusion21/fusionscript.so"
		#lib_path = "/opt/BlackmagicDesign/FusionRenderNode21/fusionscript.so"

	if not os.path.isfile(lib_path):
		print("[Fusion] [Library Does Not Exist on Disk]", lib_path)

	loader = importlib.machinery.ExtensionFileLoader("fusionscript", lib_path)
	spec = importlib.util.spec_from_loader("fusionscript", loader)
	if spec:
		bmd = importlib.util.module_from_spec(spec)
		loader.exec_module(bmd)
		if bmd:
			sys.modules[__name__] = bmd
			return bmd
		else:
			raise ImportError("[Resolve Studio] Could not locate module dependencies")
	else:
		raise ImportError("[Resolve Studio] Could not access the importlib spec loader dependencies")

def Resolve():
	app = FuScriptLib().scriptapp("Resolve")
	return app

def Fusion():
	app = FuScriptLib().scriptapp("Fusion")
	return app

# Get the Resolve and Fusion objects
resolve = Resolve()
res = resolve
app = resolve

fu = Fusion()
fusion = fu

bmd = FuScriptLib()

# Load the Lightfielder shared utility module
lightfielder_path = os.path.dirname(fu.MapPath("Scripts:/Support/lightfielder.py"))
if lightfielder_path not in sys.path:
	sys.path.append(lightfielder_path)
	# print(sys.path)
	from lightfielder import *

startTimer = datetime.datetime.now()

In [25]:
# Utility Functions for Resolve 
def GetTimeline():
	project = GetProject()
	timeline = project.GetCurrentTimeline()

	if not timeline:
		if project.GetTimelineCount() > 0:
			timeline = project.GetTimelineByIndex(1)
			project.SetCurrentTimeline(timeline)

	return timeline

def GetProject():
	# Get the current Resolve timeline
	resolve = Resolve()
	projectManager = resolve.GetProjectManager()
	project = projectManager.GetCurrentProject()
	return project

def GetMediaPool():
	resolve = Resolve()
	projectManager = resolve.GetProjectManager()
	project = projectManager.GetCurrentProject()
	mediapool = project.GetMediaPool()
	return mediapool

def GetFolder(parentFolder, childFolder, mediapool):
	if parentFolder != None:
		for folder in parentFolder.GetSubFolderList():
			if folder.GetName() == childFolder:
				return folder
		else:
			return mediapool.AddSubFolder(parentFolder, childFolder)
	else:
		return None

def GetMedia():
	clipItems = []
	fileItems = []
    
	mediapool = GetMediaPool()
	folder = mediapool.GetCurrentFolder()
	clips = folder.GetClips()
	for key in clips:
		clip = clips[key]
		if (clip.GetClipProperty("Type") == "Video") or (clip.GetClipProperty("Type") == "Video + Audio") or (clip.GetClipProperty("Type") == "Still"):
			mpFile = str(clip.GetClipProperty("File Path"))
			clipItems.append(clip)
			fileItems.append(mpFile)
	return clipItems, fileItems


# Media Pool Scripted Access

The following Python code connects to the active Resolve Studio session. It asks the current bin to report back a list of the filenames for each clip. 

In [29]:
print("[Lightfielder] " + str(LFGetVersion("Version ")))

# Open the Media page
resolve.OpenPage("media")

project = GetProject()
mediapool = project.GetMediaPool()

# Connect to the media pool and read the footage in the current bin.
clipList, mediaList = GetMedia()
if len(mediaList) == 0:
    print("[Media List][No Media in Current Bin] Please change the active bin to one that has clips in it")
else:
    print("[Media List]")
    for media in mediaList:
        print(media)

[Lightfielder] Version 26.09.08
[Media List][No Media in Current Bin] Please change the active bin to one that has clips in it


# Comments

Thanks for trying this Jupyter notebook out. It will helpfully get you on the path to effective workflow automation in Resolve Studio.